In [337]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [338]:
sale_df = pd.read_csv('../dataset/processed/processed_data.csv')

sale_df.head()

,Unnamed: 0,area,building_type,building_nature,num_bath_rooms,num_bed_rooms,price,city,locality,relaxation_amenity_count,security_amenity_count,maintenance_or_cleaning_amenity_count,social_amenity_count,expendable_amenity_count,service_staff_amenity_count,unclassify_amenity_count,division,zone
0,0,1185.0,Apartment,Residential,0.0,3.0,6100000.0,Dhaka,Khilgaon,0,1,2,0,2,0,3,Dhaka,Khilgaon
1,1,2464.0,Apartment,Residential,4.0,3.0,28900000.0,Dhaka,Dhanmondi,0,2,0,0,3,0,4,Dhaka,Dhanmondi
2,2,1140.0,Apartment,Residential,0.0,3.0,7500000.0,Dhaka,Mirpur,0,2,2,0,2,0,4,Dhaka,Mirpur
3,3,1920.0,Apartment,Residential,3.0,3.0,20000000.0,Dhaka,Bashundhara R-A,0,1,2,0,3,0,3,Dhaka,Bashundhara R/A
4,4,1445.0,Apartment,Residential,3.0,3.0,10800000.0,Dhaka,Banasree,0,0,2,0,1,0,4,Dhaka,Khilgaon


In [339]:
sale_df.shape

(12744, 18)

In [340]:
sale_df = sale_df.drop(columns=['Unnamed: 0'])

In [341]:
land_commercial_types = ['Shop', 'Office', 'Commercial Plot', 'Others', 'Residential Plot']

sale_df['bedroom_not_applicable'] = sale_df['building_type'].isin(land_commercial_types).astype(int)

In [342]:
zero_bedrooms = sale_df[sale_df['num_bed_rooms'] == 0]

zero_bedrooms['building_type'].value_counts()

building_type
Residential Plot    871
Shop                391
Office              199
Floor                79
Commercial Plot      27
Building             16
Apartment            10
Others                3
Name: count, dtype: int64

In [343]:
residential_types = [
    'Apartment',
    'Building'
]

In [344]:
floor_zero_mask = (
    (sale_df['building_type'] == 'Floor') &
    (sale_df['num_bed_rooms'] == 0)
)

print("Floor records to remove:", floor_zero_mask.sum())

sale_df = sale_df[~floor_zero_mask].copy()

print("Shape after removing Floor records:", sale_df.shape)

Floor records to remove: 79
Shape after removing Floor records: (12665, 18)


In [345]:
suspicious_mask = (
    (sale_df['num_bed_rooms'] == 0) & 
    (sale_df['building_type'].isin(residential_types))
)

In [346]:
print("Suspicious residential zero-bedroom rows:",
      suspicious_mask.sum())

Suspicious residential zero-bedroom rows: 26


In [347]:
suspicious_rows = sale_df.loc[
    suspicious_mask,
    [
        'area',
        'building_type',
        'building_nature',
        'num_bath_rooms',
        'num_bed_rooms',
        'city',
        'locality',
        'price'
    ]
]

suspicious_rows.head(20)

,area,building_type,building_nature,num_bath_rooms,num_bed_rooms,city,locality,price
10157,16000.0,Building,Commercial,0.0,0.0,Dhaka,Mirpur,120000000.0
10164,4500.0,Apartment,Commercial,0.0,0.0,Dhaka,Bashundhara R-A,58500000.0
10168,4500.0,Apartment,Commercial,0.0,0.0,Dhaka,Bashundhara R-A,55000000.0
10179,3000.0,Apartment,Commercial,0.0,0.0,Dhaka,Cantonment,22500000.0
10187,25618.0,Building,Commercial,0.0,0.0,Dhaka,Savar,85000000.0
10188,21500.0,Building,Commercial,0.0,0.0,Dhaka,Savar,220000000.0
10240,9600.0,Building,Commercial,0.0,0.0,Dhaka,Motijheel,80000000.0
10357,1100.0,Building,Commercial,0.0,0.0,Chattogram,Bakalia,7000000.0
10380,22050.0,Building,Commercial,0.0,0.0,Dhaka,Mirpur,200000000.0
10383,4500.0,Apartment,Commercial,0.0,0.0,Dhaka,Bashundhara R-A,55000000.0


In [348]:
suspicious_rows['building_type'].value_counts()

building_type
Building     16
Apartment    10
Name: count, dtype: int64

In [349]:
for btype in residential_types:

    valid_mask = (
        (sale_df['building_type'] == btype) &
        (sale_df['num_bed_rooms'] > 0)
    )

    median_bedrooms = sale_df.loc[
        valid_mask,
        'num_bed_rooms'
    ].median()

    target_mask = (
        suspicious_mask &
        (sale_df['building_type'] == btype)
    )

    sale_df.loc[
        target_mask,
        'num_bed_rooms'
    ] = median_bedrooms

In [350]:
print(
    "Suspicious residential zero-bedroom rows remaining:",
    (
        (sale_df['num_bed_rooms'] == 0) &
        sale_df['building_type'].isin(residential_types)
    ).sum()
)

Suspicious residential zero-bedroom rows remaining: 0


In [351]:
sale_df[
    sale_df['building_type'].isin(residential_types)
]['num_bed_rooms'].value_counts().sort_index()

num_bed_rooms
1.0       26
2.0     1814
3.0     8170
4.0      935
5.0       44
6.0       22
7.0       27
8.0        5
10.0       1
11.0       2
12.0       9
13.0       2
14.0      20
15.0       2
16.0       4
17.0       2
18.0       6
19.0       2
20.0       1
21.0       3
22.0       1
23.0       1
24.0       5
25.0       2
29.0       1
30.0       1
32.0       1
33.0       1
36.0       1
40.0       1
42.0       1
46.0       1
48.0       1
50.0       1
56.0       2
60.0       1
75.0       1
94.0       1
Name: count, dtype: int64

In [352]:
sale_df['log_area'] = np.log1p(sale_df['area'])

In [353]:
sale_df['total_rooms'] = (
    sale_df['num_bed_rooms'] +
    sale_df['num_bath_rooms']
)

In [354]:
sale_df['bath_bed_ratio'] = np.where(
    sale_df['bedroom_not_applicable'] == 0,
    sale_df['num_bath_rooms'] / sale_df['num_bed_rooms'],
    np.nan
)

In [355]:
sale_df['area_per_bedroom'] = np.where(
    sale_df['bedroom_not_applicable'] == 0,
    sale_df['area'] / sale_df['num_bed_rooms'],
    np.nan
)

In [356]:
amenity_columns = [
    "relaxation_amenity_count",
    "security_amenity_count",
    "maintenance_or_cleaning_amenity_count",
    "social_amenity_count",
    "expendable_amenity_count",
    "service_staff_amenity_count",
    "unclassify_amenity_count"
]

sale_df['total_amenities'] = sale_df[amenity_columns].sum(axis=1)

In [357]:
sale_df.head()

,area,building_type,building_nature,num_bath_rooms,num_bed_rooms,price,city,locality,relaxation_amenity_count,security_amenity_count,maintenance_or_cleaning_amenity_count,social_amenity_count,expendable_amenity_count,service_staff_amenity_count,unclassify_amenity_count,division,zone,bedroom_not_applicable,log_area,total_rooms,bath_bed_ratio,area_per_bedroom,total_amenities
0,1185.0,Apartment,Residential,0.0,3.0,6100000.0,Dhaka,Khilgaon,0,1,2,0,2,0,3,Dhaka,Khilgaon,0,7.078342,3.0,0.000000,395.000000,8
1,2464.0,Apartment,Residential,4.0,3.0,28900000.0,Dhaka,Dhanmondi,0,2,0,0,3,0,4,Dhaka,Dhanmondi,0,7.809947,7.0,1.333333,821.333333,9
2,1140.0,Apartment,Residential,0.0,3.0,7500000.0,Dhaka,Mirpur,0,2,2,0,2,0,4,Dhaka,Mirpur,0,7.039660,3.0,0.000000,380.000000,10
3,1920.0,Apartment,Residential,3.0,3.0,20000000.0,Dhaka,Bashundhara R-A,0,1,2,0,3,0,3,Dhaka,Bashundhara R/A,0,7.560601,6.0,1.000000,640.000000,9
4,1445.0,Apartment,Residential,3.0,3.0,10800000.0,Dhaka,Banasree,0,0,2,0,1,0,4,Dhaka,Khilgaon,0,7.276556,6.0,1.000000,481.666667,7


In [358]:
print("Final shape:", sale_df.shape)

print("\nZero bedrooms by building type:")
print(
    sale_df[sale_df['num_bed_rooms'] == 0]
    ['building_type']
    .value_counts()
)

print("\nBedroom not applicable:")
print(
    sale_df['bedroom_not_applicable'].value_counts()
)

print("\nMissing values:")
print(
    sale_df[
        [
            'num_bed_rooms',
            'num_bath_rooms',
            'bath_bed_ratio',
            'area_per_bedroom'
        ]
    ].isnull().sum()
)

Final shape: (12665, 23)

Zero bedrooms by building type:
building_type
Residential Plot    871
Shop                391
Office              199
Commercial Plot      27
Others                3
Name: count, dtype: int64

Bedroom not applicable:
bedroom_not_applicable
0    11170
1     1495
Name: count, dtype: int64

Missing values:
num_bed_rooms          0
num_bath_rooms         0
bath_bed_ratio      1495
area_per_bedroom    1495
dtype: int64


In [359]:
sale_df.to_csv('../dataset/processed/featured_data.csv')